# Test : planning à jours fixes, lissage et report des livraisons

Ce notebook part de l'algorithme de consolidation de `test_analyse.ipynb` (recopié tel quel en partie 1) et construit le planning de livraison avec des **jours fixes** (SF-04) : chaque couple fournisseur × usine livre toujours le ou les mêmes jours de la semaine. Il ajoute un **lissage** des livraisons par usine (SF-08), appliqué seulement quand une semaine est trop déséquilibrée, et produit le **planning complet** sur toute la durée.

```
Algorithme de base (date du camion = premier besoin)
        │
        ▼
Jours fixes : chaque couple teste ses profils possibles (1 jour ou 2 jours par semaine)
et garde le moins cher ; le camion part le dernier jour fixe avant son premier besoin
        │
        ▼
Contrôle par usine, semaine par semaine : un jour s'écarte-t-il de plus de 35 % de la moyenne ?
        │ non ──► rien à faire
        │ oui
        ▼
Lissage : on change le profil (jours fixes) de certains couples, au moindre surcoût,
jusqu'à repasser dans la tolérance
        │
        ▼
Report : les lignes qui peuvent attendre partent avec le camion du jour fixe suivant
(semaine suivante comprise), quand il a la place et que ça ne coûte rien
        │
        ▼
Planning complet par usine + bilan
```

**Hypothèses :**
- **lignes de la quarantaine** (partie 1) : elles sont livrées elles aussi (SF-12). Une ligne sans poids ne peut pas être consolidée : elle a son **propre camion**, le jour de son besoin, chiffré dans la tranche **> 4 000 kg** (estimation haute). Un fournisseur sans distance est chiffré à **> 650 km** (estimation haute). La ligne de plus de 24,5 t est exclue en attendant la réponse du client ;
- **jamais de retard, au plus 7 jours d'avance** (SF-06) ; les livraisons restent dans l'horizon du fichier (09/11 → 18/12), les jours ouvrés ;
- **profils** : 1 jour fixe ou 2 jours fixes par semaine. À coût égal, on prend le profil avec le moins de jours (SF-05). Les deux variantes (1 jour seulement, 1 ou 2 jours) sont comparées ;
- **camions de démarrage** : les besoins du début d'horizon qui tombent avant le premier jour fixe du couple ne peuvent pas être livrés avant le 09/11. Ils partent à la date de leur besoin, sont comptés dans le coût, mais restent hors du contrôle et du lissage, qui ne peut pas les déplacer ;
- **charge d'une usine** = nombre de camions reçus chaque jour. Une semaine est **hors tolérance** si un jour s'écarte de plus de 35 % (paramètre `SEUIL_LISSAGE`) de la moyenne de la semaine, avec au moins 1 camion de marge (les semaines de moins de 5 camions ne sont pas contrôlées). Le seuil est choisi en partie 6 ;
- la **dernière semaine** (S51) n'est pas contrôlée : elle est incomplète (l'horizon s'arrête le 18/12).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SILVER_CLEAN_PATH = "../data/silver/released_schedule_cleaned.csv"
BASELINE_PATH = "../data/baseline/baseline_camion.csv"   # baseline du milestone 2 : une ligne par camion

POIDS_MAX = 24500   # kg
VOLUME_MAX = 110.4  # m³

# constantes pour le calcul du coût de transport
FREIGHT_VALUE_MARGIN = 2.3923  # +239,23 %
DIESEL_SURCHARGE = 1.1943

couts = pd.read_csv("../data/bronze/rate_sheet.csv", index_col="km_max")
TRANCHES_KM = [int(km) for km in couts.index if not km.startswith(">")]    # 30, 50, ..., 650
TRANCHES_KG = [int(kg) for kg in couts.columns if not kg.startswith(">")]  # 50, 100, ..., 4000

# distance de chaque fournisseur (le fichier a un BOM -> utf-8-sig)
distances = pd.read_csv("../data/bronze/distances.csv", encoding="utf-8-sig").set_index("supplier_id")["distance"]


def tranche(valeur, tranches):
    """Première tranche >= valeur (on majore), sinon la tranche '>max'."""
    for borne in tranches:
        if valeur <= borne:
            return str(borne)
    return f">{tranches[-1]}"


def cout_transport(poids, distance):
    """Prix d'un camion en € = coût de la grille × (1 + marge) × diesel."""
    cout_base = couts.loc[tranche(distance, TRANCHES_KM), tranche(poids, TRANCHES_KG)]
    return float(round(cout_base * (1 + FREIGHT_VALUE_MARGIN) * DIESEL_SURCHARGE, 2))


def nombre(x):
    """24500 -> '24 500', 110.4 -> '110,4' (format français)."""
    return f"{x:,.1f}".replace(",", " ").replace(".", ",").removesuffix(",0")


def pourcent(x):
    """0.956 -> '95,6 %'."""
    return f"{x:.1%}".replace(".", ",").replace("%", " %")


def eur(x):
    """142964.96 -> '142 965 €'."""
    return f"{x:,.0f} €".replace(",", " ")

import itertools

JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]
SEUIL_LISSAGE = 0.35       # écart max toléré entre un jour et la moyenne de la semaine (voir partie 6)
MIN_CAMIONS_SEMAINE = 5    # semaines plus petites (moins d'un camion par jour) : pas contrôlées

# Données manquantes : estimations hautes
QUARANTAINE_PATH = "../data/silver/released_schedule_quarantine.csv"
DISTANCE_PAR_DEFAUT = TRANCHES_KM[-1] + 1  # 651 km -> tranche "> 650 km"
POIDS_PAR_DEFAUT = TRANCHES_KG[-1] + 1     # 4 001 kg -> tranche "> 4 000 kg"
SANS_DISTANCE = sorted(distances.index[distances.isna()])
distances = distances.fillna(DISTANCE_PAR_DEFAUT)

## 1. Données et algorithme de base

### Lignes de la quarantaine

Le pipeline du milestone 1 a mis de côté 139 lignes dans `released_schedule_quarantine.csv`. Ce sont pourtant de vrais besoins, qui doivent être livrés (SF-12). Elles sont traitées comme dans la baseline du milestone 2 :

| Cas | Traitement | Coût |
|---|---|---|
| **Poids inconnu** (`MISSING_WEIGHT`) | impossible à consolider : **un camion réservé par ligne**, le jour du besoin | tranche **> 4 000 kg** (estimation haute) |
| **Ligne de plus de 24,5 t** (`OVER_TRUCK_WEIGHT`, SUP_00074 → PLT_003 le 15/12, 26,9 t) | **exclue pour le moment**, en attente de la réponse du client | – |
| **Fournisseur sans distance** (`NO_DISTANCE`), poids connu | **consolidée normalement** avec les autres lignes | distance **> 650 km** (estimation haute) |

Les camions réservés ne bougent pas, mais ils arrivent aux quais : ils sont comptés dans la charge des usines, et le lissage s'organise autour.

In [ ]:
quarantaine = pd.read_csv(QUARANTAINE_PATH, parse_dates=["need_date"])
sans_poids = quarantaine["weight"].isna()
trop_lourd = quarantaine["weight"] > POIDS_MAX
a_consolider = ~sans_poids & ~trop_lourd          # poids connu (ex. fournisseur sans distance)

# Camions réservés : un par ligne sans poids, le jour du besoin (la ligne de plus de 24,5 t est exclue)
camions_reserves, lignes_reserves = [], []
for numero, (idx, ligne) in enumerate(quarantaine[sans_poids].iterrows(), start=1):
    camions_reserves.append({
        "numero": numero, "plant": ligne["plant"], "supplier_id": ligne["supplier_id"], "date": ligne["need_date"],
        "demarrage": False, "nb_lignes": 1, "poids": np.nan, "volume": ligne["volume"],
        "cout": cout_transport(POIDS_PAR_DEFAUT, distances[ligne["supplier_id"]]), "motif": "poids inconnu"})
    lignes_reserves.append({**ligne.drop("flags").to_dict(), "numero": numero,
                            "date_livraison": ligne["need_date"], "avance_jours": 0})
camions_reserves = pd.DataFrame(camions_reserves)
lignes_reserves = pd.DataFrame(lignes_reserves)
COUT_RESERVE, NB_RESERVE = camions_reserves["cout"].sum(), len(camions_reserves)


def avec_reserves(camions):
    """Ajoute les camions réservés (lignes de la quarantaine) à une liste de camions."""
    return pd.concat([camions, camions_reserves.drop(columns="numero")], ignore_index=True)


print(f"{len(quarantaine)} lignes en quarantaine :")
print(f"  - {sans_poids.sum()} sans poids -> {NB_RESERVE} camions réservés, {eur(COUT_RESERVE)}")
print(f"  - {trop_lourd.sum()} de plus de 24,5 t -> exclue, en attente de la réponse du client : "
      + ", ".join(f"{r.supplier_id} -> {r.plant} le {r.need_date:%d/%m} ({nombre(r.weight / 1000)} t)" for r in quarantaine[trop_lourd].itertuples()))
print(f"  - {a_consolider.sum()} avec un poids connu (fournisseurs {sorted(quarantaine.loc[a_consolider, 'supplier_id'].unique())}) -> consolidées")
print(f"Fournisseurs sans distance (chiffrés à > {TRANCHES_KM[-1]} km) : {len(SANS_DISTANCE)}, "
      f"dont dans le planning : {sorted(set(SANS_DISTANCE) & set(quarantaine['supplier_id']))}")
camions_reserves.groupby(["plant", "motif"]).agg(camions=("numero", "size"), cout=("cout", "sum")).round(2)

### Algorithme de base (copie de `test_analyse.ipynb`)

Recopié de `test_analyse.ipynb`, avec **une seule modification** : les lignes de la quarantaine dont le poids est connu sont ajoutées au planning à consolider. L'algorithme sert de point de départ et de référence pour mesurer ce que coûtent les jours fixes et le lissage.

In [ ]:
CAP_W, CAP_V = POIDS_MAX, VOLUME_MAX   # capacité d'un camion : kg, m³
FENETRE = pd.Timedelta(days=7)        # un camion prend les besoins jusqu'à sa date de livraison + 7 jours

sch = pd.read_csv(SILVER_CLEAN_PATH, parse_dates=["need_date"])
N_SILVER = len(sch)   # lignes 0 à N_SILVER - 1 : silver ; les suivantes viennent de la quarantaine
# Ajout (test) : lignes de la quarantaine dont le poids est connu, consolidées comme les autres
sch = pd.concat([sch, quarantaine.loc[a_consolider, sch.columns]], ignore_index=True)
assert (sch["weight"] <= CAP_W).all() and (sch["volume"] <= CAP_V).all()  # chaque ligne rentre seule dans un camion

# Tri : par couple (usine, fournisseur), puis date, puis poids du plus lourd au plus léger
sch = sch.sort_values(["plant", "supplier_id", "need_date", "weight"], ascending=[True, True, True, False])

camion_de_la_ligne = {}   # index de la ligne du schedule -> n° du camion
date_du_camion = {}       # n° du camion -> date de livraison
n_camion = 0

for (plant, supplier), lignes_couple in sch.groupby(["plant", "supplier_id"], sort=False):
    restantes = list(lignes_couple.itertuples())   # lignes du couple pas encore chargées (triées)

    while restantes:
        # Nouveau camion : il livre à la date de la première ligne restante (la plus ancienne)
        n_camion += 1
        date_camion = restantes[0].need_date
        date_du_camion[n_camion] = date_camion
        poids, volume = 0.0, 0.0
        i = 0

        while i < len(restantes):
            ligne = restantes[i]
            if ligne.need_date > date_camion + FENETRE:
                # Hors fenêtre : les lignes suivantes sont encore plus tardives -> camion fermé
                break
            if poids + ligne.weight > CAP_W or volume + ligne.volume > CAP_V:
                # Ne rentre pas dans le camion : on essaie la ligne suivante
                i += 1
                continue
            # Rentre : ajoutée au camion, retirée des lignes restantes, on repart du début
            poids += ligne.weight
            volume += ligne.volume
            camion_de_la_ligne[ligne.Index] = n_camion
            restantes.pop(i)
            i = 0
        # Fin de la boucle (hors fenêtre ou plus rien ne rentre) : le camion est fermé,
        # le while extérieur en ouvre un nouveau s'il reste des lignes

# ---------- Résultats ----------
sch["camion_id"] = sch.index.map(camion_de_la_ligne)
sch["date_livraison"] = sch["camion_id"].map(date_du_camion)
sch["avance_jours"] = (sch["need_date"] - sch["date_livraison"]).dt.days

# Une ligne par camion
camions_opti = (sch.groupby(["camion_id", "plant", "supplier_id", "date_livraison"])
                   .agg(poids=("weight", "sum"), volume=("volume", "sum"), nb_lignes=("weight", "size"),
                        avance_max=("avance_jours", "max"))
                   .reset_index())

# Contrôles
assert sch["camion_id"].notna().all()                                   # chaque ligne est dans un camion
assert (camions_opti["poids"] <= CAP_W).all() and (camions_opti["volume"] <= CAP_V).all()
assert sch["avance_jours"].between(0, FENETRE.days).all()              # jamais en retard, au plus 7 j d'avance

print(f"{len(sch)} lignes -> {len(camions_opti)} camions (baseline : 1 603 camions)")
print(f"Poids médian d'un camion : {camions_opti['poids'].median():,.0f} kg".replace(",", " "))
print(f"Camions au-delà de 4 000 kg : {(camions_opti['poids'] > 4000).sum()}")
print(f"Avance moyenne d'une ligne : {sch['avance_jours'].mean():.1f} j")

In [ ]:
# Coût de chaque camion : baseline (milestone 2) et algorithme de base
camions_algo = camions_opti.rename(columns={"date_livraison": "date"})
camions_base = pd.read_csv(BASELINE_PATH, parse_dates=["date"])
# La baseline du milestone 2 contient déjà les lignes de la quarantaine : ses camions réservés sont les mêmes
# que ceux d'ici, comptés à part (COUT_RESERVE) pour toutes les versions
assert camions_base["reserve"].sum() == NB_RESERVE
camions_base = camions_base[~camions_base["reserve"]]
for camions in (camions_base, camions_algo):
    camions["cout"] = [cout_transport(p, distances[s]) for p, s in zip(camions["poids"], camions["supplier_id"])]

COUT_BASELINE = camions_base["cout"].sum()
COUT_ALGO = camions_algo["cout"].sum()
print(f"Baseline           : {len(camions_base)} camions, {eur(COUT_BASELINE)} (+ {NB_RESERVE} camions réservés, {eur(COUT_RESERVE)})")
print(f"Algorithme de base : {len(camions_algo)} camions, {eur(COUT_ALGO)} (+ les mêmes camions réservés)")
print(f"Économie : {eur(COUT_BASELINE - COUT_ALGO)}, soit {pourcent(1 - COUT_ALGO / COUT_BASELINE)} hors camions réservés "
      f"et {pourcent((COUT_BASELINE - COUT_ALGO) / (COUT_BASELINE + COUT_RESERVE))} camions réservés compris")

## 2. Contrôle de l'équilibre de la semaine, par usine

Pour chaque usine et chaque semaine, on compte les camions reçus chaque jour. Une semaine est **hors tolérance** si un jour s'écarte de la moyenne de la semaine de plus de **35 %** (paramètre `SEUIL_LISSAGE`), avec au moins **1 camion de marge** : une usine qui reçoit 5 camions dans la semaine ne peut pas faire mieux qu'un par jour. Les semaines de moins de 5 camions (moins d'un camion par jour en moyenne) ne sont pas contrôlées : elles ne chargent pas les quais. Si une semaine au moins est hors tolérance, l'usine doit être lissée.

In [ ]:
besoins = sch[["need_date", "supplier_id", "material", "plant", "unloading_point",
               "schedule_quantity", "weight", "volume"]].copy()   # trié : couple, date, poids décroissant
DEBUT, FIN = besoins["need_date"].min(), besoins["need_date"].max()   # horizon : lundi 09/11 -> vendredi 18/12
SEMAINES = sorted(besoins["need_date"].dt.isocalendar().week.unique())
NB_SEMAINES = len(SEMAINES)
SEMAINES_CONTROLEES = SEMAINES[:-1]                     # la dernière semaine est incomplète
CONTROLE = [SEMAINES.index(s) for s in SEMAINES_CONTROLEES]
USINES = sorted(besoins["plant"].unique())


def case(date):
    """(n° de semaine dans l'horizon, jour de la semaine) d'une date."""
    return SEMAINES.index(date.isocalendar().week), date.day_of_week


def charge_depuis_camions(camions, plant):
    """Camions reçus par une usine : une ligne par semaine, une colonne par jour (lundi -> vendredi)."""
    charge = np.zeros((NB_SEMAINES, 5))
    for d in camions.loc[camions["plant"] == plant, "date"]:
        if d.isocalendar().week in SEMAINES and d.day_of_week < 5:
            charge[case(d)] += 1
    return charge


def charges_camions(camions):
    """Charge de chaque usine à partir d'une liste de camions (colonnes plant et date)."""
    return {plant: charge_depuis_camions(camions, plant) for plant in USINES}


def ecart(charge_jours):
    """Écart du jour le plus éloigné de la moyenne, en % de la moyenne (0.35 = 35 %)."""
    charge_jours = np.asarray(charge_jours, dtype=float)
    moyenne = charge_jours.mean()
    return 0.0 if moyenne == 0 else float(np.abs(charge_jours - moyenne).max() / moyenne)


def hors_tolerance(charge_semaines, seuil=None):
    """Pour chaque semaine : un jour s'écarte-t-il de la moyenne de plus du seuil (et d'au moins 1 camion) ?
    Les semaines de moins de MIN_CAMIONS_SEMAINE camions ne sont pas contrôlées."""
    seuil = SEUIL_LISSAGE if seuil is None else seuil
    moyenne = charge_semaines.mean(axis=1, keepdims=True)
    tolerance = np.maximum(seuil * moyenne, 1.0)
    depasse = (np.abs(charge_semaines - moyenne) > tolerance + 1e-9).any(axis=1)
    return depasse & (charge_semaines.sum(axis=1) >= MIN_CAMIONS_SEMAINE)


def tableau_controle(charges):
    """Par usine : camions moyens par jour, pire semaine et décision de lissage."""
    lignes = []
    for plant, charge in charges.items():
        hors = hors_tolerance(charge[CONTROLE])
        lignes.append({"plant": plant, **dict(zip(JOURS, charge.mean(axis=0))),
                       "Pire semaine": max(ecart(charge[w]) for w in CONTROLE),
                       "Semaines hors tolérance": int(hors.sum()),
                       "Lissage nécessaire": bool(hors.any())})
    return pd.DataFrame(lignes).set_index("plant")


def ecarts_par_semaine(charges):
    """Écart de chaque usine, semaine par semaine (✗ = hors tolérance ; la dernière semaine n'est pas contrôlée)."""
    table = {}
    for plant, charge in charges.items():
        hors = hors_tolerance(charge)
        table[plant] = [pourcent(ecart(charge[w])) + (" ✗" if hors[w] and w in CONTROLE else "")
                        + f" ({int(charge[w].sum())} cam.)" for w in range(NB_SEMAINES)]
    return pd.DataFrame(table, index=[f"S{s}" for s in SEMAINES]).T


def heatmap_avant_apres(charges_avant, charges_apres, titre, etapes=("avant lissage", "après lissage")):
    """Camions reçus par semaine × jour pour chaque usine, avant (en haut) et après (en bas)."""
    fig, axes = plt.subplots(2, len(USINES), figsize=(16, 7.5), sharex=True, sharey=True)
    for col, plant in enumerate(USINES):
        for ligne, (etape, charges) in enumerate(zip(etapes, [charges_avant, charges_apres])):
            ax = axes[ligne, col]
            charge = charges[plant]
            hors = hors_tolerance(charge)
            im = ax.imshow(charge, cmap="YlGnBu", vmin=0, vmax=20, aspect="auto")
            for w in range(NB_SEMAINES):
                for j in range(5):
                    v = int(charge[w, j])
                    ax.text(j, w, v, ha="center", va="center", color="white" if v > 12 else "black",
                            fontweight="bold" if hors[w] and w in CONTROLE else "normal")
            ax.set_xticks(range(5), [j[:3] for j in JOURS])
            ax.set_yticks(range(NB_SEMAINES), [f"S{s}" for s in SEMAINES])
            ax.set_title(f"{plant} {etape}\n{int(hors[CONTROLE].sum())} semaine(s) hors tolérance", fontsize=10)
    fig.colorbar(im, ax=axes, label="Camions reçus", shrink=0.8)
    fig.suptitle(f"{titre}\n(en gras : semaine hors tolérance, S{SEMAINES[-1]} non contrôlée)", fontsize=13, y=1.02)
    plt.show()


format_controle = {**{j: "{:.1f}" for j in JOURS}, "Pire semaine": pourcent}
avant_algo = tableau_controle(charges_camions(avec_reserves(camions_algo)))   # camions réservés compris
display(avant_algo.style.format(format_controle))
ecarts_par_semaine(charges_camions(avec_reserves(camions_algo)))

#### Analyse : contrôle de l'algorithme de base

Camions réservés compris, l'algorithme de base n'est équilibré dans aucune usine : **les 20 semaines contrôlées sont hors tolérance**, avec des écarts allant jusqu'à 237 % (PLT_002 en S47). Un couple qui a des besoins tous les jours livre tous les 8 jours, donc un jour plus tard chaque semaine : les gros couples finissent par livrer le même jour. Cette dérive empêche aussi d'avoir des jours de livraison fixes.

## 3. Jours fixes : 1 ou 2 jours par couple

Même boucle que l'algorithme de base, une seule différence : quand un camion s'ouvre, il ne part pas à la date du premier besoin restant, mais au **dernier jour fixe du couple avant ce besoin** (ou le jour même). Il prend ensuite tous les besoins jusqu'à sa date + 7 jours, dans la limite de 24,5 t et 110,4 m³.

```
Couple au profil « mardi + vendredi »
besoin restant le plus ancien : jeudi 19/11  ──►  camion le mardi 17/11 (prend les besoins du 17/11 au 24/11)
besoin restant le plus ancien : mercredi 25/11 ──►  camion le mardi 24/11 (prend les besoins du 24/11 au 01/12)
besoin restant le plus ancien : samedi 05/12   ──►  camion le vendredi 04/12 ...
```

Chaque couple teste ses **15 profils** (5 profils à 1 jour, 10 profils à 2 jours) et on garde le moins cher (à coût égal : le moins de camions, puis le moins de jours, SF-05). Deux variantes sont construites :
- **1 jour fixe** : seulement les 5 profils à 1 jour ;
- **1 ou 2 jours fixes** : les 15 profils.

Un couple qui n'a pas de besoin une semaine saute simplement sa livraison, et si le besoin dépasse un camion, plusieurs camions partent le même jour.

In [ ]:
PROFILS = [(j,) for j in range(5)] + list(itertools.combinations(range(5), 2))
nom_profil = lambda profil: " + ".join(JOURS[j] for j in profil)


def date_camion(besoin, profil):
    """Dernier jour du profil avant ou égal au besoin ; s'il tombe avant l'horizon -> date du besoin (démarrage)."""
    date = max(besoin - pd.Timedelta(days=(besoin.day_of_week - j) % 7) for j in profil)
    return (besoin, True) if date < DEBUT else (date, False)


def consolider_couple(lignes, profil):
    """Algorithme de base avec la date du camion calée sur les jours fixes du couple (lignes triées date, poids ↓)."""
    restantes = list(lignes.itertuples())
    camions = []
    while restantes:
        date, demarrage = date_camion(restantes[0].need_date, profil)
        poids, volume, contenu = 0.0, 0.0, []
        i = 0
        while i < len(restantes):
            ligne = restantes[i]
            if ligne.need_date > date + FENETRE:
                break                                   # hors fenêtre de 7 jours : camion fermé
            if poids + ligne.weight > CAP_W or volume + ligne.volume > CAP_V:
                i += 1                                  # ne rentre pas : ligne suivante
                continue
            poids += ligne.weight
            volume += ligne.volume
            contenu.append(ligne.Index)
            restantes.pop(i)
            i = 0
        camions.append({"date": date, "demarrage": demarrage, "poids": poids, "volume": volume, "lignes": contenu})
    return pd.DataFrame(camions)


# Toutes les options : chaque couple × chaque profil
scenarios = {}
for (plant, supplier), lignes_couple in besoins.groupby(["plant", "supplier_id"]):
    for profil in PROFILS:
        camions = consolider_couple(lignes_couple, profil)
        camions["cout"] = [cout_transport(p, distances[supplier]) for p in camions["poids"]]
        scenarios[(plant, supplier, profil)] = camions
cout_option = {cle: c["cout"].sum() for cle, c in scenarios.items()}
nb_camions_option = {cle: len(c) for cle, c in scenarios.items()}
charge_option = {(p, s, prof): charge_depuis_camions(c[~c["demarrage"]].assign(plant=p), p)   # hors démarrage
                 for (p, s, prof), c in scenarios.items()}


def choix_moins_cher(profils_autorises):
    """Profil le moins cher de chaque couple (à égalité : moins de camions, puis moins de jours, puis ordre)."""
    choix = {}
    for cle in sorted((k for k in scenarios if k[2] in profils_autorises),
                      key=lambda k: (k[0], k[1], round(cout_option[k], 2), nb_camions_option[k], len(k[2]), k[2])):
        choix.setdefault(cle[:2], cle[2])
    return choix


cout_choix = lambda choix: sum(cout_option[(p, s, prof)] for (p, s), prof in choix.items())
VARIANTES = {"1 jour fixe": [p for p in PROFILS if len(p) == 1], "1 ou 2 jours fixes": PROFILS}
choix_initiaux = {nom: choix_moins_cher(profils) for nom, profils in VARIANTES.items()}

print(f"{len(scenarios)} options évaluées ({besoins.groupby(['plant', 'supplier_id']).ngroups} couples × {len(PROFILS)} profils)")
for nom, choix in choix_initiaux.items():
    nb = sum(nb_camions_option[(p, s, prof)] for (p, s), prof in choix.items())
    dem = sum(int(scenarios[(p, s, prof)]["demarrage"].sum()) for (p, s), prof in choix.items())
    print(f"{nom:<20}: {nb} camions (dont {dem} de démarrage), {eur(cout_choix(choix))} "
          f"(+{eur(cout_choix(choix) - COUT_ALGO)} vs algorithme de base), "
          f"couples à 2 jours : {sum(len(prof) == 2 for prof in choix.values())}")

#### Analyse : jours fixes

Coûts **hors camions réservés** (les mêmes 134 camions dans toutes les versions) :

| Variante | Camions | dont démarrage | Coût | Surcoût vs algorithme de base | Couples à 2 jours |
|---|---|---|---|---|---|
| 1 jour fixe | 611 | 63 | 79 859 € | +3 852 € | 0 |
| 1 ou 2 jours fixes | 573 | 52 | 77 242 € | **+1 235 €** | 60 |

Avec un seul jour fixe, un couple qui a des besoins tous les jours livre tous les 7 jours au lieu de tous les 8 dans l'algorithme de base, et 63 camions de démarrage sont nécessaires en première semaine. **Autoriser un deuxième jour fixe divise le surcoût par 3** :
- le camion part le dernier des deux jours fixes avant le premier besoin, donc plus près de ce besoin : sa fenêtre de 7 jours couvre plus de besoins à venir ;
- avec deux jours dans la semaine, un jour fixe tombe plus souvent tôt en première semaine, d'où moins de camions de démarrage (52 au lieu de 63).

## 4. Contrôle et lissage par couple

Pour chaque variante et chaque usine, on contrôle les semaines (camions de démarrage exclus). Si une semaine au moins est hors tolérance, on répète :
1. on teste tous les changements de profil possibles d'un couple de l'usine (autre jour, ou autre paire de jours) ;
2. on garde ceux qui rendent les semaines plus régulières (la somme, sur les semaines contrôlées, des carrés des écarts à la moyenne de la semaine baisse) ;
3. on applique celui qui coûte le moins par unité d'amélioration (les changements gratuits d'abord, le plus efficace en premier) ;

jusqu'à ce qu'aucune semaine ne soit hors tolérance. Les usines déjà équilibrées ne sont pas touchées.

In [ ]:
CHARGE_RESERVEE = charges_camions(camions_reserves)   # camions réservés : charge fixe, que le lissage ne déplace pas


def charges_choix(choix, fond=None):
    """Charge de chaque usine pour un choix de profils (hors démarrage), avec un fond de charge fixe
    (par défaut : les camions réservés)."""
    fond = CHARGE_RESERVEE if fond is None else fond
    return {plant: fond[plant] + sum(charge_option[(p, s, prof)] for (p, s), prof in choix.items() if p == plant)
            for plant in USINES}


def dispersion(charge):
    """Somme, sur les semaines contrôlées, des carrés des écarts à la moyenne de la semaine."""
    charge = charge[CONTROLE]
    return float(((charge - charge.mean(axis=1, keepdims=True)) ** 2).sum())


def lisser_par_couple(choix, plant, profils_autorises, seuil=None, bavard=True, fond=None):
    """Change le profil de couples de `plant` jusqu'à ce qu'aucune semaine contrôlée ne soit hors tolérance."""
    choix = dict(choix)
    charge = charges_choix(choix, fond)[plant]
    journal = []
    while hors_tolerance(charge[CONTROLE], seuil).any():
        dispersion_actuelle = dispersion(charge)
        meilleur = None
        for (p, s), prof in choix.items():
            if p != plant:
                continue
            for prof2 in profils_autorises:
                if prof2 == prof:
                    continue
                nouvelle = charge - charge_option[(p, s, prof)] + charge_option[(p, s, prof2)]
                gain = dispersion_actuelle - dispersion(nouvelle)
                if gain <= 1e-9:
                    continue
                surcout = cout_option[(p, s, prof2)] - cout_option[(p, s, prof)]
                cle = (max(surcout, 0) / gain, -gain)
                if meilleur is None or cle < meilleur["cle"]:
                    meilleur = {"cle": cle, "supplier_id": s, "de": prof, "vers": prof2, "surcout": surcout,
                                "charge": nouvelle}
        if meilleur is None:
            if bavard:
                print(f"{plant} : plus aucun changement de profil n'améliore les semaines")
            break
        choix[(plant, meilleur["supplier_id"])] = meilleur["vers"]
        charge = meilleur["charge"]
        journal.append({"plant": plant, "supplier_id": meilleur["supplier_id"], "de": nom_profil(meilleur["de"]),
                        "vers": nom_profil(meilleur["vers"]), "surcout": round(meilleur["surcout"], 2),
                        "semaines_hors_tolerance_apres": int(hors_tolerance(charge[CONTROLE], seuil).sum())})
    return choix, journal


def lisser(choix, profils_autorises, seuil=None, bavard=True, fond=None):
    """Lisse toutes les usines qui ont au moins une semaine hors tolérance ; retourne le choix et le journal."""
    choix, journal = dict(choix), []
    for plant in USINES:
        if hors_tolerance(charges_choix(choix, fond)[plant][CONTROLE], seuil).any():
            choix, journal_u = lisser_par_couple(choix, plant, profils_autorises, seuil, bavard, fond)
            journal += journal_u
    return choix, pd.DataFrame(journal)


choix_lisses, journaux = {}, {}
for nom, profils in VARIANTES.items():
    choix_lisses[nom], journaux[nom] = lisser(choix_initiaux[nom], profils)
    choix, journal = choix_lisses[nom], journaux[nom]
    avant = tableau_controle(charges_choix(choix_initiaux[nom]))["Semaines hors tolérance"].sum()
    apres = tableau_controle(charges_choix(choix))["Semaines hors tolérance"].sum()
    print(f"{nom:<20}: {len(journal)} changements de profil, surcoût {eur(cout_choix(choix) - cout_choix(choix_initiaux[nom]))}, "
          f"-> {eur(cout_choix(choix))} ; semaines hors tolérance {avant} -> {apres} (sur {len(USINES) * len(CONTROLE)})")

# Variante principale : la moins chère après lissage
PRINCIPALE = min(VARIANTES, key=lambda nom: cout_choix(choix_lisses[nom]))
print(f"\nVariante principale : {PRINCIPALE}")
display(tableau_controle(charges_choix(choix_initiaux[PRINCIPALE])).style.format(format_controle))
display(tableau_controle(charges_choix(choix_lisses[PRINCIPALE])).style.format(format_controle))
ecarts_par_semaine(charges_choix(choix_lisses[PRINCIPALE]))

In [ ]:
# Journal des changements de profil de la variante principale (traçabilité)
journaux[PRINCIPALE]

In [ ]:
heatmap_avant_apres(charges_choix(choix_initiaux[PRINCIPALE]), charges_choix(choix_lisses[PRINCIPALE]),
                    f"{PRINCIPALE} : camions reçus par semaine et par jour, hors démarrage")

#### Analyse : lissage par couple

| Variante | Changements de profil | Surcoût | Coût après lissage (hors réservés) | Semaines hors tolérance |
|---|---|---|---|---|
| 1 jour fixe | 93 | 1 117 € | 80 975 € | 20 → 2 |
| **1 ou 2 jours fixes** | 127 | 1 535 € | **78 777 €** | 20 → 1 |

Le lissage coûte environ deux fois plus que sans les camions réservés. Ces 134 camions arrivent le jour même du besoin et ne bougent pas : 52 vont à PLT_002, et 24 à PLT_003 pour le seul SUP_00129. Le lissage doit donc placer les autres couples autour.

La variante à 1 ou 2 jours reste **2 198 € moins chère** que la variante à 1 jour après lissage : c'est la **variante principale**. Il ne reste qu'une semaine hors tolérance, PLT_002 en S50 : une petite semaine de 6 camions, dont 4 le même mardi.

## 5. Report des lignes sur le jour fixe suivant

Avec des jours fixes, un camion part le **dernier jour fixe avant son premier besoin** : le jour fixe suivant tombe donc toujours après ce besoin. Un camion ne peut jamais être reporté entier sans livrer ce besoin en retard. En revanche, il transporte souvent des lignes dont le besoin tombe **le jour fixe suivant ou après** (jusqu'à 7 jours plus tard) : celles-là peuvent partir avec le camion du jour fixe suivant, y compris la semaine d'après.

Pour chaque couple, on reporte ces lignes (des plus lourdes aux plus légères) dans le camion du couple qui part le jour fixe suivant, **s'il existe et s'il a la place** : on n'ajoute jamais de camion. Ensuite, si un couple a plusieurs camions le même jour et qu'ils tiennent désormais dans un seul, on les **regroupe** : un camion de moins ce jour-là. Le report n'est gardé pour un couple que s'il **ne coûte rien** (le prix dépend du poids de chaque camion).

Les camions de démarrage peuvent aussi reporter leurs lignes vers le premier jour fixe du couple.

In [ ]:
def prochain_jour_fixe(date, profil):
    """Premier jour du profil strictement après `date`."""
    return min(date + pd.Timedelta(days=(j - date.day_of_week - 1) % 7 + 1) for j in profil)


def construire_planning(choix):
    """Planning (une ligne par camion, avec la liste de ses lignes) pour un choix de profils."""
    camions = pd.concat([scenarios[(p, s, prof)].assign(plant=p, supplier_id=s, profil=[prof] * len(scenarios[(p, s, prof)]))
                         for (p, s), prof in choix.items()], ignore_index=True)
    return camions


def reporter_lignes(camions):
    """Report des lignes sur le jour fixe suivant + regroupement des camions d'un même jour, couple par couple,
    gardé seulement s'il ne coûte rien. Retourne les nouveaux camions et le journal par couple."""
    besoin, poids, volume = besoins["need_date"], besoins["weight"], besoins["volume"]
    charge_ok = lambda ls: poids[ls].sum() <= CAP_W + 1e-9 and volume[ls].sum() <= CAP_V + 1e-9
    resultat, journal = [], []
    for (plant, supplier), groupe in camions.groupby(["plant", "supplier_id"], sort=False):
        profil = groupe["profil"].iloc[0]
        cout = lambda ls: cout_transport(poids[ls].sum(), distances[supplier])
        cam = [{"date": r.date, "demarrage": r.demarrage, "lignes": list(r.lignes)}
               for r in groupe.sort_values("date").itertuples()]
        cout_avant, nb_avant = sum(cout(c["lignes"]) for c in cam), len(cam)
        reportees = 0
        for c in cam:                                           # 1) report vers le jour fixe suivant
            suivant = prochain_jour_fixe(c["date"], profil)
            cibles = [u for u in cam if u["date"] == suivant and not u["demarrage"]]
            for l in sorted((l for l in c["lignes"] if besoin[l] >= suivant), key=lambda l: -poids[l]):
                for u in cibles:
                    if charge_ok(u["lignes"] + [l]):
                        c["lignes"].remove(l)
                        u["lignes"].append(l)
                        reportees += 1
                        break
        for date in sorted({c["date"] for c in cam}):            # 2) regroupement des camions d'un même jour
            du_jour = [c for c in cam if c["date"] == date and c["lignes"]]
            for a, b in itertools.combinations(du_jour, 2):
                if a["lignes"] and b["lignes"] and charge_ok(a["lignes"] + b["lignes"]):
                    a["lignes"] += b["lignes"]
                    b["lignes"] = []
        apres = [c for c in cam if c["lignes"]]
        cout_apres = sum(cout(c["lignes"]) for c in apres)
        garde = cout_apres <= cout_avant + 0.005
        if not garde:
            apres = [{"date": r.date, "demarrage": r.demarrage, "lignes": list(r.lignes)} for r in groupe.itertuples()]
        journal.append({"plant": plant, "supplier_id": supplier, "lignes_reportees": reportees if garde else 0,
                        "camions_en_moins": nb_avant - len(apres), "gain": round(cout_avant - cout_apres, 2) if garde else 0.0,
                        "report_refuse_car_plus_cher": not garde and reportees > 0})
        resultat += [{**c, "plant": plant, "supplier_id": supplier, "profil": profil} for c in apres]
    return pd.DataFrame(resultat), pd.DataFrame(journal)


def finaliser(camions, attendues=None):
    """Recalcule le contenu, le coût et les dates de chaque camion ; retourne le planning et l'affectation des lignes.
    `attendues` : lignes qui doivent toutes être livrées (par défaut : tous les besoins)."""
    attendues = besoins if attendues is None else attendues
    camions = camions.sort_values(["plant", "date", "supplier_id"]).reset_index(drop=True)
    camions["camion_id"] = range(1, len(camions) + 1)
    affectation = camions[["camion_id", "date", "lignes"]].explode("lignes").set_index("lignes")
    lignes = attendues.join(affectation.rename(columns={"date": "date_livraison"}))
    lignes["avance_jours"] = (lignes["need_date"] - lignes["date_livraison"]).dt.days
    contenu = lignes.groupby("camion_id").agg(poids=("weight", "sum"), volume=("volume", "sum"), nb_lignes=("weight", "size"))
    planning = camions.drop(columns=[c for c in ["lignes", "poids", "volume", "cout"] if c in camions]).join(contenu, on="camion_id")
    planning["cout"] = [cout_transport(p, distances[s]) for p, s in zip(planning["poids"], planning["supplier_id"])]
    planning["jours_fixes"] = planning["profil"].map(nom_profil)
    planning["jour"] = planning["date"].dt.day_of_week.map(dict(enumerate(JOURS)))
    planning["semaine"] = planning["date"].dt.isocalendar().week
    planning = planning[["camion_id", "plant", "supplier_id", "jours_fixes", "date", "jour", "semaine", "demarrage",
                         "nb_lignes", "poids", "volume", "cout", "profil"]]
    # Contrôles : chaque ligne une fois, jamais en retard, au plus 7 j d'avance, capacités, jours fixes
    assert lignes["camion_id"].notna().all() and lignes.index.is_unique and len(affectation) == len(attendues)
    assert lignes["avance_jours"].between(0, FENETRE.days).all()
    assert (planning["poids"] <= CAP_W + 1e-6).all() and (planning["volume"] <= CAP_V + 1e-6).all()
    assert all(d.day_of_week in prof or dem for d, prof, dem in zip(planning["date"], planning["profil"], planning["demarrage"]))
    assert planning["date"].between(DEBUT, FIN).all()
    return planning, lignes


plannings = {}
for nom in VARIANTES:
    plannings[f"{nom}"] = finaliser(construire_planning(choix_initiaux[nom]))
    plannings[f"{nom} + lissage"] = finaliser(construire_planning(choix_lisses[nom]))
camions_report, journal_report = reporter_lignes(construire_planning(choix_lisses[PRINCIPALE]))
plannings[f"{PRINCIPALE} + lissage + report"] = finaliser(camions_report)
planning, lignes_planning = plannings[f"{PRINCIPALE} + lissage + report"]
avant_report, lignes_avant_report = plannings[f"{PRINCIPALE} + lissage"]

print(f"Lignes reportées sur le jour fixe suivant : {journal_report['lignes_reportees'].sum()} "
      f"({journal_report['lignes_reportees'].gt(0).sum()} couples)")
print(f"Camions en moins (regroupements) : {journal_report['camions_en_moins'].sum()}, "
      f"gain : {eur(journal_report['gain'].sum())}, reports refusés car plus chers : {journal_report['report_refuse_car_plus_cher'].sum()} couples")
print(f"Avance moyenne d'une ligne : {nombre(lignes_avant_report['avance_jours'].mean())} j -> "
      f"{nombre(lignes_planning['avance_jours'].mean())} j ; "
      f"lignes livrées le jour de leur besoin : {(lignes_avant_report['avance_jours'] == 0).sum()} -> {(lignes_planning['avance_jours'] == 0).sum()}")
apres_report = tableau_controle(charges_camions(avec_reserves(planning[~planning["demarrage"]])))
print(f"Semaines hors tolérance (hors démarrage) après report : {apres_report['Semaines hors tolérance'].sum()}")
journal_report[journal_report["camions_en_moins"] > 0]

#### Analyse : report sur le jour fixe suivant

Le report fait partir **112 lignes** (14 couples) avec le camion du jour fixe suivant au lieu de 3 à 7 jours en avance. Les lignes livrées le jour même de leur besoin passent de 652 à 762, et l'avance moyenne de 3,3 à 3,2 jours. Le gain est faible (72 €) et **aucun camion n'est supprimé**. Pour 15 autres couples, le report a été refusé car il coûtait plus cher : le poids ajouté fait monter le camion du jour suivant dans une tranche plus chère que ce que le camion de départ économise.

C'est la limite du report avec des jours fixes : un camion part toujours le dernier jour fixe avant son premier besoin, donc **il ne peut jamais être reporté entier** sans retard. Il part de toute façon, avec ses autres lignes, et le jour chargé ne perd pas de camion. Avec des jours fixes, ce qui fait vraiment baisser un pic, c'est le **changement de profil d'un couple** (partie 4). Le report sert surtout à réduire le stock.

## 6. Choix du seuil de lissage

**La question** : quel seuil choisir ? Plus le seuil est bas, plus on lisse : les semaines sont plus régulières, mais des couples sont éloignés de leur profil le moins cher, donc le coût monte.

**La méthode** : on refait tout le calcul (lissage puis report) pour plusieurs seuils, du « sans lissage » jusqu'à 15 %, et on compare trois choses :

| Indicateur | Ce qu'il mesure | On veut |
|---|---|---|
| **Surcoût du lissage** | ce que le lissage ajoute au coût de transport, par rapport à « sans lissage » | le plus bas possible |
| **Jour le plus chargé** | le plus grand nombre de camions reçus un même jour par une usine, sur toute la période | le plus bas possible |
| **Irrégularité moyenne** | en moyenne, de combien le jour le plus éloigné de la semaine s'écarte de la moyenne de cette semaine (ex. 50 % : avec 6 camions par jour en moyenne, un jour en reçoit 9 ou 3) | le plus bas possible |

Les deux derniers indicateurs sont calculés de la même façon pour tous les seuils, sur les semaines contrôlées et hors camions de démarrage : on peut donc comparer les seuils entre eux.

In [ ]:
SEUILS_TESTES = [None, 1.0, 0.8, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15]


def indicateurs_equilibre(camions):
    # Jour le plus chargé et irrégularité moyenne (pondérée par les camions de chaque semaine),
    # sur les semaines contrôlées, hors camions de démarrage
    ecarts, poids, pic = [], [], 0
    for charge in charges_camions(camions[~camions["demarrage"]]).values():
        for w in CONTROLE:
            if charge[w].sum() > 0:
                ecarts.append(ecart(charge[w]))
                poids.append(charge[w].sum())
                pic = max(pic, charge[w].max())
    return int(pic), np.average(ecarts, weights=poids)


etude = []
for seuil in SEUILS_TESTES:
    if seuil is None:
        choix = choix_initiaux[PRINCIPALE]
    else:
        choix, _ = lisser(choix_initiaux[PRINCIPALE], VARIANTES[PRINCIPALE], seuil, bavard=False)
    camions_s, _ = reporter_lignes(construire_planning(choix))
    planning_s, _ = finaliser(camions_s)
    pic, irregularite = indicateurs_equilibre(avec_reserves(planning_s))
    etude.append({"Seuil": "sans lissage" if seuil is None else f"{seuil:.0%}".replace("%", " %"), "seuil": seuil,
                  "Coût total": planning_s["cout"].sum() + COUT_RESERVE, "Camions": len(planning_s) + NB_RESERVE,
                  "Jour le plus chargé (camions)": pic, "Irrégularité moyenne": irregularite})
etude = pd.DataFrame(etude)
etude.insert(3, "Surcoût du lissage", etude["Coût total"] - etude.loc[0, "Coût total"])

retenu = lambda ligne: ["background-color: #fde9c9"] * len(ligne) if ligne.name == f"{SEUIL_LISSAGE:.0%}".replace("%", " %") else [""] * len(ligne)
(etude.drop(columns="seuil").set_index("Seuil").style
 .format({"Coût total": eur, "Surcoût du lissage": eur, "Irrégularité moyenne": pourcent})
 .apply(retenu, axis=1))

In [ ]:
# ---------- Trois indicateurs selon le seuil (de gauche à droite : on lisse de plus en plus) ----------
x = np.arange(len(etude))
i_retenu = etude.index[etude["seuil"] == SEUIL_LISSAGE][0]
graphes = [("Surcoût du lissage", "Ce que coûte le lissage (€)", "royalblue", eur),
           ("Jour le plus chargé (camions)", "Jour le plus chargé (camions)", "darkorange", lambda v: f"{v:.0f}"),
           ("Irrégularité moyenne", "Irrégularité moyenne des semaines", "seagreen", pourcent)]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharex=True)
for ax, (colonne, titre, couleur, fmt) in zip(axes, graphes):
    valeurs = etude[colonne] * (100 if colonne == "Irrégularité moyenne" else 1)
    ax.plot(x, valeurs, marker="o", color=couleur)
    for xi, (v, brut) in enumerate(zip(valeurs, etude[colonne])):
        if xi == 0 or brut != etude[colonne].iloc[xi - 1]:          # pas d'étiquette répétée
            ax.annotate(fmt(brut), (xi, v), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
    ax.axvline(i_retenu, color="red", linestyle="--", alpha=0.6)
    ax.text(i_retenu, ax.get_ylim()[1], " seuil retenu", color="red", va="top", fontsize=9)
    ax.set_title(titre)
    ax.set_xticks(x, etude["Seuil"], rotation=45, ha="right")
    ax.set_xlabel("Seuil (à droite : on lisse davantage)")
    ax.margins(y=0.15)
    ax.grid(alpha=0.3)
axes[2].yaxis.set_major_formatter(lambda v, _: f"{v:.0f} %")
fig.suptitle("Plus on baisse le seuil, plus on lisse : ce que ça coûte et ce que ça apporte", fontsize=13)
plt.tight_layout()
plt.show()

#### Analyse : choix du seuil

**Comment lire les graphiques** : de gauche à droite, on baisse le seuil, donc on lisse de plus en plus. Le coût monte (à gauche), tandis que le jour le plus chargé et l'irrégularité baissent (au milieu et à droite).

**Ce qu'on en retient (camions réservés compris) :**
- **Le premier lissage fait l'essentiel** : dès le seuil de 100 %, pour 1 065 €, le jour le plus chargé passe de 30 à 14 camions et l'irrégularité de 95 % à 47 %.
- **Le jour le plus chargé ne descend jamais sous 14 camions** : c'est la semaine S48 de PLT_002, la plus lourde (66 camions, environ 13 par jour, camions réservés compris). Ce n'est plus un pic, mais une semaine chargée partout.
- **Jusqu'à 40 %, la régularité reste bon marché** : de 100 % à 40 %, l'irrégularité passe de 47 % à 23 % pour 188 € de plus seulement.
- **En dessous de 40 %, elle devient chère** : passer à 35 % coûte 299 € pour 3,6 points d'irrégularité, soit environ 80 € par point, contre moins de 10 € par point entre 100 % et 40 %.

| Seuil | Coût total | Surcoût du lissage | Jour le plus chargé | Irrégularité moyenne |
|---|---|---|---|---|
| sans lissage | 110 659 € | 0 € | 30 camions | 95 % |
| 50 % | 111 863 € | 1 204 € | 14 camions | 24 % |
| **40 %** | **111 912 €** | **1 253 €** | **14 camions** | **23 %** |
| 35 % (actuel) | 112 210 € | 1 552 € | 14 camions | 19 % |
| 15 % | 112 578 € | 1 919 € | 14 camions | 14 % |

**Recommandation : avec les camions réservés, le meilleur compromis passe de 35 % à 40 %.** Ça économise environ 300 € par rapport à 35 %, pour une irrégularité moyenne de 23 % au lieu de 19 %. Le notebook garde 35 % (`SEUIL_LISSAGE`) en attendant ta décision.

## 7. Planning complet sur toute la durée, par usine

Planning final (variante principale, lissée, avec report), **camions réservés compris**. Une ligne par camion : son couple, les jours fixes du couple, sa date, son contenu et son coût. Les camions de démarrage et les camions réservés (lignes de la quarantaine) sont signalés.

In [ ]:
# Planning complet : camions optimisés + camions réservés (numérotés à la suite)
decalage = int(planning["camion_id"].max())
reserves = camions_reserves.assign(camion_id=camions_reserves["numero"] + decalage,
                                   jours_fixes="réservé : " + camions_reserves["motif"])
planning_complet = (pd.concat([planning.assign(reserve=False), reserves.assign(reserve=True)], ignore_index=True)
                    .drop(columns=["numero", "motif"]))
planning_complet["jour"] = planning_complet["date"].dt.day_of_week.map(dict(enumerate(JOURS)))
planning_complet["semaine"] = planning_complet["date"].dt.isocalendar().week
planning_complet = planning_complet.sort_values(["plant", "date", "supplier_id"]).reset_index(drop=True)
lignes_complet = pd.concat([lignes_planning, lignes_reserves.assign(camion_id=lignes_reserves["numero"] + decalage)
                            .drop(columns="numero")], ignore_index=True)
assert lignes_complet["camion_id"].isin(planning_complet["camion_id"]).all()
print(f"Planning complet : {len(planning_complet)} camions ({len(planning)} optimisés + {NB_RESERVE} réservés), "
      f"{eur(planning_complet['cout'].sum())}")

# ---------- Vue par usine : camions reçus par semaine × jour (démarrage et camions réservés compris) ----------
fig, axes = plt.subplots(1, len(USINES), figsize=(16, 4.4), sharey=True)
for ax, plant in zip(axes, USINES):
    charge = charge_depuis_camions(planning_complet, plant)
    im = ax.imshow(charge, cmap="YlGnBu", vmin=0, vmax=20, aspect="auto")
    for w in range(NB_SEMAINES):
        for j in range(5):
            ax.text(j, w, int(charge[w, j]), ha="center", va="center", color="white" if charge[w, j] > 12 else "black")
    ax.set_xticks(range(5), [j[:3] for j in JOURS])
    ax.set_yticks(range(NB_SEMAINES), [f"S{s}" for s in SEMAINES])
    ax.set_title(f"{plant} : {int(charge.sum())} camions, {eur(planning_complet.loc[planning_complet['plant'] == plant, 'cout'].sum())}", fontsize=10)
fig.colorbar(im, ax=axes, label="Camions reçus")
fig.suptitle("Planning complet : camions reçus par semaine et par jour (démarrage et camions réservés compris)", fontsize=13, y=1.03)
plt.show()

# Profil de chaque couple (sortie 1)
profils_couples = (planning.groupby(["plant", "supplier_id", "jours_fixes"])
                   .agg(camions=("camion_id", "size"), camions_demarrage=("demarrage", "sum"),
                        poids_total=("poids", "sum"), cout=("cout", "sum")).reset_index())
print("Couples par nombre de jours fixes :",
      profils_couples["jours_fixes"].str.count(r"\+").add(1).value_counts().sort_index().rename(lambda n: f"{n} jour(s)").to_dict())
profils_couples.head(10)

In [ ]:
# Détail du planning complet d'une usine (une ligne par camion) : changer l'usine ici
USINE = "PLT_004"
planning_complet.loc[planning_complet["plant"] == USINE].drop(columns=["plant", "profil"]).round({"poids": 1, "volume": 2})

## 8. Bilan

Comparaison de toutes les versions. **Toutes comprennent les mêmes camions réservés** (lignes de la quarantaine) : ils s'ajoutent au coût et à la charge des quais, sans changer l'économie en euros. L'équilibre est mesuré sur **tous les camions reçus** (camions de démarrage compris) et sur les 5 semaines contrôlées de chaque usine (20 au total). La semaine S46 est comptée à part : son lundi 09/11 reçoit les camions de démarrage, qui ne peuvent être ni avancés (l'horizon commence ce jour-là) ni retardés.

In [ ]:
def nb_hors_tolerance(camions, semaines):
    return sum(int(hors_tolerance(charge[semaines]).sum()) for charge in charges_camions(camions).values())



versions = {"Baseline": (camions_base, None), "Algorithme de base": (camions_algo, sch)}
versions.update({nom: (p, l) for nom, (p, l) in plannings.items()})
COUT_BASELINE_TOTAL = COUT_BASELINE + COUT_RESERVE
bilan = pd.DataFrame([{
    "": nom, "Camions": len(c) + NB_RESERVE, "Coût (€)": c["cout"].sum() + COUT_RESERVE,
    "Économie vs baseline (€)": COUT_BASELINE - c["cout"].sum(),
    "Économie vs baseline (%)": (COUT_BASELINE - c["cout"].sum()) / COUT_BASELINE_TOTAL,
    "Part de l'économie de l'algo conservée": (COUT_BASELINE - c["cout"].sum()) / (COUT_BASELINE - COUT_ALGO),
    "Avance moyenne (j)": np.nan if l is None else l["avance_jours"].mean(),
    "Couples à 2 jours": c.drop_duplicates(["plant", "supplier_id"])["profil"].map(len).eq(2).sum() if "profil" in c else np.nan,
    f"Semaines hors tolérance (sur {len(USINES) * len(CONTROLE)})": nb_hors_tolerance(avec_reserves(c), CONTROLE),
    "dont S46": nb_hors_tolerance(avec_reserves(c), CONTROLE[:1]),
} for nom, (c, l) in versions.items()]).set_index("")
bilan.style.format({"Coût (€)": eur, "Économie vs baseline (€)": eur, "Économie vs baseline (%)": pourcent,
                    "Avance moyenne (j)": "{:.1f}",
                    "Couples à 2 jours": "{:.0f}", "Part de l'économie de l'algo conservée": pourcent}, na_rep="–")

#### Analyse : bilan

Le planning final (**1 ou 2 jours fixes, lissé à 35 %, avec report**) compte **736 camions** (602 optimisés + 134 réservés) pour **112 210 €**, dont **33 506 € de camions réservés** (30 % du coût total). L'économie est de **65 110 €** par rapport à la baseline : **-36,7 %** camions réservés compris, -45,3 % sur la partie optimisable. On garde **96,0 % de l'économie** de l'algorithme de base : les jours fixes et le lissage coûtent 2 698 €.

- Par rapport à la variante à 1 jour fixe lissée (114 481 €), on gagne **2 271 €**, et l'avance moyenne baisse de 3,6 à 3,2 jours.
- L'équilibre est atteint : 4 semaines hors tolérance sur 20, contre 20 pour l'algorithme de base et 12 pour la baseline. 3 sont les **lundis 09/11** de PLT_001, PLT_002 et PLT_004 (18, 23 et 18 camions), chargés par les camions de démarrage. La 4e est la petite semaine S50 de PLT_002.
- 66 couples livrent sur 2 jours possibles, et 107 sur un seul jour.
- La ligne de plus de 24,5 t (SUP_00074, 26,9 t le 15/12) n'est comptée dans aucune version, en attendant la réponse du client.

### Levier : la qualité des données

Pour mesurer ce que coûtent les données manquantes, on refait le calcul **sans les lignes de la quarantaine**, c'est-à-dire avec les seules lignes du silver, dont les données sont complètes : pas de camions réservés, pas de fournisseur sans distance. Le planning final est recalculé de la même façon (profils, lissage sans le fond des camions réservés, report).

In [ ]:
# Version sans les lignes de la quarantaine : seules les lignes du silver
lignes_silver = besoins[besoins.index < N_SILVER]
couples_quarantaine = set(map(tuple, besoins.loc[besoins.index >= N_SILVER, ["plant", "supplier_id"]].values))
assert not couples_quarantaine & set(map(tuple, lignes_silver[["plant", "supplier_id"]].values))  # couples séparés
FOND_VIDE = {plant: np.zeros((NB_SEMAINES, 5)) for plant in USINES}

choix_sans = {cle: prof for cle, prof in choix_initiaux[PRINCIPALE].items() if cle not in couples_quarantaine}
choix_sans, _ = lisser(choix_sans, VARIANTES[PRINCIPALE], bavard=False, fond=FOND_VIDE)
camions_sans, _ = reporter_lignes(construire_planning(choix_sans))
planning_sans, lignes_sans = finaliser(camions_sans, attendues=lignes_silver)
base_sans = camions_base[camions_base["source"] == "silver"]
algo_sans = camions_algo[~camions_algo["camion_id"].isin(sch.loc[sch.index >= N_SILVER, "camion_id"])]

comparaison = []
for nom, sans, avec in [("Baseline", base_sans, avec_reserves(camions_base)),
                        ("Algorithme de base", algo_sans, avec_reserves(camions_algo)),
                        ("Planning final (jours fixes, lissage, report)", planning_sans, avec_reserves(planning))]:
    comparaison.append({"": nom, "Camions sans": len(sans), "Coût sans": sans["cout"].sum(),
                        "Camions avec": len(avec), "Coût avec": avec["cout"].sum()})
comparaison = pd.DataFrame(comparaison).set_index("")
for version in ["sans", "avec"]:
    base = comparaison.loc["Baseline", f"Coût {version}"]
    comparaison[f"Économie {version}"] = (base - comparaison[f"Coût {version}"]) / base
comparaison = comparaison[["Camions sans", "Coût sans", "Économie sans", "Camions avec", "Coût avec", "Économie avec"]]

print(f"Camions réservés (poids inconnu) : {NB_RESERVE} camions, {eur(COUT_RESERVE)}, soit "
      f"{pourcent(COUT_RESERVE / comparaison.loc['Planning final (jours fixes, lissage, report)', 'Coût avec'])} "
      f"du coût du planning final")
comparaison.style.format({c: eur for c in comparaison if c.startswith("Coût")}
                         | {c: pourcent for c in comparaison if c.startswith("Économie")})

#### Analyse : qualité des données

| | Sans les lignes de la quarantaine | Avec (camions réservés compris) |
|---|---|---|
| Baseline | 142 965 € | 177 321 € |
| Planning final | 77 443 € | 112 210 € |
| **Économie** | **65 522 € (-45,8 %)** | **65 110 € (-36,7 %)** |

**L'optimisation fait le même travail dans les deux cas** : environ 65 000 € d'économie. Mais avec les lignes de la quarantaine, le pourcentage tombe de 45,8 % à 36,7 %, parce que **134 camions réservés (33 506 €) s'ajoutent au coût et ne peuvent pas être optimisés**. Ils représentent 30 % du coût du planning final.

C'est un **levier d'optimisation important, mais hors de notre main** : ces lignes n'ont pas de poids dans le fichier du client. Tant que le poids manque, on ne peut ni regrouper ces lignes avec d'autres ni choisir leur camion, et on doit les chiffrer en estimation haute (tranche > 4 000 kg). Si le client renseignait ces poids, ces lignes seraient consolidées comme les autres. L'économie possible ne peut pas être chiffrée sans les vrais poids, mais elle porte sur 33 506 €, soit 30 % du coût du planning final.

Les 4 lignes du fournisseur sans distance (SUP_00130) sont, elles, consolidées normalement : seule leur distance est estimée (> 650 km).

## 9. Export

Les tables du planning final sont écrites dans `data/gold/test_planning/` :
- `planning.csv` : une ligne par camion, toute la durée, toutes les usines, camions réservés compris (colonne `reserve`) ;
- `contenu_camions.csv` : une ligne par ligne de besoin, avec son camion et sa date de livraison ;
- `profils_couples.csv` : les jours fixes de chaque couple.

In [ ]:
from pathlib import Path

SORTIE = Path("../data/gold/test_planning")
SORTIE.mkdir(parents=True, exist_ok=True)
planning_complet.drop(columns="profil").to_csv(SORTIE / "planning.csv", index=False)
lignes_complet.sort_values(["camion_id", "need_date"]).to_csv(SORTIE / "contenu_camions.csv", index=False)
profils_couples.to_csv(SORTIE / "profils_couples.csv", index=False)
print(*sorted(p.name for p in SORTIE.iterdir()), sep="\n")

## Conclusion

Le moteur produit le **planning complet** de chaque usine sur toute la durée, avec des **jours fixes** par couple (1 ou 2 jours possibles par semaine). Il prend en compte les **lignes de la quarantaine** comme la baseline du milestone 2 : camions réservés pour les lignes sans poids, distance maximale pour le fournisseur sans distance. La ligne de plus de 24,5 t est exclue en attendant la réponse du client. Il **lisse** chaque usine seulement quand une semaine dépasse le seuil, sans jamais livrer en retard ni plus de 7 jours en avance.

| Version (camions réservés compris) | Coût | Économie vs baseline | Semaines hors tolérance |
|---|---|---|---|
| Baseline | 177 321 € | – | 12 / 20 |
| Algorithme de base (pas de jours fixes) | 109 513 € | 67 807 € (-38,2 %) | 20 / 20 |
| 1 jour fixe + lissage | 114 481 € | 62 839 € (-35,4 %) | 5 / 20 |
| **1 ou 2 jours fixes + lissage 35 % + report** | **112 210 €** | **65 110 € (-36,7 %)** | **4 / 20** |

Sans les lignes de la quarantaine, le même planning économise **45,8 %** : l'écart vient des 134 camions réservés pour des lignes sans poids (33 506 €), que l'optimisation ne peut pas toucher.

**Seuil de lissage** : avec les camions réservés, l'étude de la partie 6 recommande **40 %** plutôt que 35 %. On économise environ 300 € pour une régularité presque aussi bonne. Le notebook garde 35 % en attendant la décision.

**Limites et pistes**
- **Qualité des données** : retrouver les poids manquants est le plus gros levier restant (30 % du coût final en camions réservés). Il dépend du client.
- **Ligne de plus de 24,5 t** : à intégrer quand le client aura répondu.
- **Camions de démarrage** (lundi 09/11) : ils viennent du début de l'horizon et ne peuvent être ni avancés ni retardés.
- **Report** : avec des jours fixes, il ne peut que rapprocher des lignes de leur besoin, sans supprimer de camion.
- **Lissage glouton** : il change un profil à la fois. Une recherche plus large (OR-Tools) pourrait aller plus loin.
- **Étude du seuil sur 6 semaines** : à refaire sur un autre planning pour confirmer le compromis.